# 05 — 多源数据分析

四个独立模块，可分开运行：

| Module | 主题 | 先决条件 |
|--------|------|----------|
| A | 宏观全景（PMI + 指数、CPI/PPI）| 宏观数据 + 宽基指数 |
| B | 北向资金与大盘 | 北向数据 + 上证 |
| C | 个股多维筛选 | 财务 + 资金流 + K 线 |
| D | C++ 特征验证（雷达图）| trade_cli 路径已修复 |

In [ ]:
# ── 初始化 ────────────────────────────────────────────────────────────────────
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
DATA_ROOT = PROJECT_ROOT / 'data'

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import date, timedelta

TODAY = date.today()
print(f'Project: {PROJECT_ROOT}')
print(f'Today:   {TODAY}')

---
## Module A — 宏观全景

In [ ]:
# A-1: PMI + 沪深300 双轴图
macro_dir = DATA_ROOT / 'macro'
idx_dir = DATA_ROOT / 'index'

# 加载 PMI
pmi_path = macro_dir / 'pmi.parquet'
hs300_path = idx_dir / '000300_SH.parquet'

if not pmi_path.exists() or not hs300_path.exists():
    print('⚠ 缺少 PMI 或沪深300数据，请先运行:\n  ./trade data macro sync\n  ./trade data index sync')
else:
    pmi = pd.read_parquet(pmi_path)
    hs300 = pd.read_parquet(hs300_path)

    # 标准化日期列
    date_col_pmi = next((c for c in ['month', 'date', 'ann_date'] if c in pmi.columns), None)
    pmi['_date'] = pd.to_datetime(pmi[date_col_pmi], errors='coerce')
    pmi = pmi.dropna(subset=['_date']).sort_values('_date')

    hs300['_date'] = pd.to_datetime(hs300['date'])
    hs300 = hs300.sort_values('_date')

    # PMI 值列
    pmi_col = next((c for c in ['pmi', 'value', 'manufacturing_pmi', 'nmc_pmi'] if c in pmi.columns), None)

    fig = make_subplots(specs=[[{'secondary_y': True}]])

    if pmi_col:
        pmi_vals = pd.to_numeric(pmi[pmi_col], errors='coerce')
        fig.add_trace(go.Bar(x=pmi['_date'], y=pmi_vals, name='PMI 制造业',
                             marker_color=['#ef5350' if v >= 50 else '#42a5f5' for v in pmi_vals.fillna(49)],
                             opacity=0.7), secondary_y=False)
        # 50 分界线
        fig.add_hline(y=50, line_dash='dash', line_color='gray', secondary_y=False)

    fig.add_trace(go.Scatter(x=hs300['_date'], y=hs300['close'], name='沪深300',
                             line=dict(color='#ff7043', width=2)), secondary_y=True)

    fig.update_layout(title='PMI 制造业 vs 沪深300（月度）', hovermode='x unified')
    fig.update_yaxes(title_text='PMI', secondary_y=False)
    fig.update_yaxes(title_text='沪深300 收盘', secondary_y=True)
    fig.show()

In [ ]:
# A-2: CPI / PPI 叠加折线图
cpi_path = macro_dir / 'cpi.parquet'
ppi_path = macro_dir / 'ppi.parquet'

if not cpi_path.exists() and not ppi_path.exists():
    print('⚠ 缺少 CPI/PPI 数据，请先运行: ./trade data macro sync')
else:
    fig = go.Figure()

    for label, path, color in [('CPI 同比%', cpi_path, '#42a5f5'), ('PPI 同比%', ppi_path, '#ff7043')]:
        if not path.exists():
            continue
        df = pd.read_parquet(path)
        date_col = next((c for c in ['month', 'date', 'ann_date'] if c in df.columns), None)
        if not date_col:
            continue
        df['_date'] = pd.to_datetime(df[date_col], errors='coerce')
        val_col = next((c for c in ['nt_val', 'yoy', 'value', 'ppi_yoy', 'cpi_yoy'] if c in df.columns), None)
        if not val_col:
            continue
        df = df.dropna(subset=['_date']).sort_values('_date')
        vals = pd.to_numeric(df[val_col], errors='coerce')
        fig.add_trace(go.Scatter(x=df['_date'], y=vals, name=label, line=dict(color=color)))

    fig.add_hline(y=0, line_dash='dash', line_color='gray')
    fig.update_layout(title='CPI / PPI 同比（通胀/通缩观测）', hovermode='x unified')
    fig.show()

---
## Module B — 北向资金与大盘

In [ ]:
# B-1: 北向资金净流入 vs 上证指数 双轴图
nb_dir = DATA_ROOT / 'northbound'
sh_path = idx_dir / '000001_SH.parquet'

nb_files = list(nb_dir.glob('*.parquet')) if nb_dir.exists() else []

if not nb_files:
    print('⚠ 缺少北向资金数据，请先运行: ./trade data northbound sync --start 2023-01-01')
elif not sh_path.exists():
    print('⚠ 缺少上证综指数据，请先运行: ./trade data index sync')
else:
    nb = pd.concat([pd.read_parquet(f) for f in nb_files], ignore_index=True)
    date_col = next((c for c in ['trade_date', 'date'] if c in nb.columns), None)
    nb['_date'] = pd.to_datetime(nb[date_col], errors='coerce')

    # 北向净流入列（沪深通合计）
    net_col = next((c for c in ['north_money', 'net_buy', 'hgt_amount', 'north_net'] if c in nb.columns), None)
    if net_col is None and {'hgt_amount', 'sgt_amount'}.issubset(nb.columns):
        nb['net_flow'] = pd.to_numeric(nb['hgt_amount'], errors='coerce').fillna(0) + \
                         pd.to_numeric(nb['sgt_amount'], errors='coerce').fillna(0)
        net_col = 'net_flow'

    sh = pd.read_parquet(sh_path)
    sh['_date'] = pd.to_datetime(sh['date'])

    # 近2年
    cutoff = pd.Timestamp(TODAY - timedelta(days=730))
    nb_recent = nb[nb['_date'] >= cutoff].sort_values('_date')
    sh_recent = sh[sh['_date'] >= cutoff].sort_values('_date')

    fig = make_subplots(specs=[[{'secondary_y': True}]])
    if net_col:
        vals = pd.to_numeric(nb_recent[net_col], errors='coerce') / 1e8  # 亿元
        fig.add_trace(go.Bar(
            x=nb_recent['_date'], y=vals, name='北向净流入(亿元)',
            marker_color=['#ef5350' if v >= 0 else '#42a5f5' for v in vals.fillna(0)],
            opacity=0.7,
        ), secondary_y=False)
    fig.add_trace(go.Scatter(x=sh_recent['_date'], y=sh_recent['close'],
                             name='上证综指', line=dict(color='#ff7043', width=2)), secondary_y=True)
    fig.update_layout(title='北向资金净流入 vs 上证综指（近2年）', hovermode='x unified')
    fig.update_yaxes(title_text='北向净流入(亿元)', secondary_y=False)
    fig.update_yaxes(title_text='上证收盘', secondary_y=True)
    fig.show()

In [ ]:
# B-2: 当日北向净流入 → 后5日指数涨跌幅散点分析
if nb_files and sh_path.exists() and net_col:
    merged = pd.merge(
        nb[['_date', net_col]].dropna(),
        sh[['_date', 'close']].rename(columns={'close': 'sh_close'}),
        on='_date', how='inner'
    ).sort_values('_date').reset_index(drop=True)

    merged['nb_bn'] = pd.to_numeric(merged[net_col], errors='coerce') / 1e8
    merged['fwd5_ret'] = merged['sh_close'].pct_change(5).shift(-5) * 100
    merged = merged.dropna()

    fig = px.scatter(
        merged, x='nb_bn', y='fwd5_ret',
        title='北向当日净流入 → 后5日上证涨跌幅',
        labels={'nb_bn': '当日北向净流入(亿元)', 'fwd5_ret': '后5日涨跌幅(%)'},
        trendline='ols', opacity=0.6,
    )
    fig.show()
else:
    print('（B-2 需要北向数据）')

---
## Module C — 个股多维筛选

In [ ]:
# 筛选参数（修改这里）
MIN_ROE = 0.10          # ROE 下限
MAX_PB  = 10.0          # PB 上限
MIN_FUND_FLOW_RATIO = 0.0  # 大单净流入比下限（0=不过滤）
LOOKBACK_DAYS = 5          # 资金流均值窗口

fund_dir = DATA_ROOT / 'fundamental'
ff_dir   = DATA_ROOT / 'fund_flow'
kline_dir = DATA_ROOT / 'kline'

if not fund_dir.exists() or not list(fund_dir.glob('*.parquet')):
    print('⚠ 缺少财务数据，请先运行: ./trade data fundamental sync')
elif not ff_dir.exists() or not list(ff_dir.glob('*.parquet')):
    print('⚠ 缺少资金流向数据，请先运行: ./trade data fund-flow sync')
else:
    # 加载财务数据
    fund_dfs = []
    for f in fund_dir.glob('*.parquet'):
        try:
            df = pd.read_parquet(f)
            df['symbol'] = f.stem
            fund_dfs.append(df)
        except Exception:
            pass

    if fund_dfs:
        fund_all = pd.concat(fund_dfs, ignore_index=True)
        date_col = next((c for c in ['ann_date', 'end_date', 'date'] if c in fund_all.columns), None)
        if date_col:
            fund_all['_date'] = pd.to_datetime(fund_all[date_col], errors='coerce')
            # 取每支股票最新一条
            fund_latest = fund_all.sort_values('_date').groupby('symbol').last().reset_index()
        else:
            fund_latest = fund_all.groupby('symbol').last().reset_index()

        roe_col = next((c for c in ['roe', 'roe_yearly', 'roe_dt'] if c in fund_latest.columns), None)
        pb_col  = next((c for c in ['pb', 'pb_mrq'] if c in fund_latest.columns), None)

        if roe_col:
            fund_latest[roe_col] = pd.to_numeric(fund_latest[roe_col], errors='coerce')
            screened = fund_latest[fund_latest[roe_col] >= MIN_ROE * 100]
            if pb_col:
                fund_latest[pb_col] = pd.to_numeric(fund_latest[pb_col], errors='coerce')
                screened = screened[screened[pb_col] <= MAX_PB]
            print(f'财务筛选通过: {len(screened)} / {len(fund_latest)} 只')
            display(screened[['symbol', roe_col] + ([pb_col] if pb_col else [])].head(20))
        else:
            print(f'财务数据可用列: {fund_latest.columns.tolist()}')
    else:
        print('无法加载财务数据')

In [ ]:
# C-2: 气泡图（ROE × 大单净流入 × 行业）
if 'screened' in dir() and len(screened) > 0 and ff_dir.exists():
    ff_dfs = []
    for f in list(ff_dir.glob('*.parquet'))[:200]:
        try:
            df = pd.read_parquet(f)
            df['symbol'] = f.stem
            ff_dfs.append(df.tail(LOOKBACK_DAYS))
        except Exception:
            pass

    if ff_dfs:
        ff_all = pd.concat(ff_dfs, ignore_index=True)
        # 大单净流入比
        net_col = next((c for c in ['buy_lg_amount', 'net_mf_amount', 'large_order_net_ratio'] if c in ff_all.columns), None)
        if net_col:
            ff_agg = ff_all.groupby('symbol')[net_col].mean().reset_index(name='avg_net_flow')
            merged = screened.merge(ff_agg, on='symbol', how='inner')
            merged = merged[merged['avg_net_flow'] >= MIN_FUND_FLOW_RATIO]

            try:
                from trade_py.analysis.knowledge_graph import SW, SW_NAMES_ZH
                import sqlite3
                db_path = DATA_ROOT / '.metadata' / 'trade.db'
                if db_path.exists():
                    with sqlite3.connect(str(db_path)) as conn:
                        df_ind = pd.read_sql('SELECT symbol, industry FROM instruments', conn)
                    df_ind['industry_name'] = df_ind['industry'].apply(
                        lambda x: SW_NAMES_ZH.get(SW(int(x)), '未知') if x > 0 else '未映射'
                    )
                    merged = merged.merge(df_ind[['symbol', 'industry_name']], on='symbol', how='left')
                    merged['industry_name'] = merged['industry_name'].fillna('未知')
            except Exception:
                merged['industry_name'] = '未知'

            print(f'综合筛选通过: {len(merged)} 只')

            if not merged.empty and roe_col in merged.columns:
                fig = px.scatter(
                    merged, x=roe_col, y='avg_net_flow',
                    color='industry_name' if 'industry_name' in merged.columns else None,
                    hover_data=['symbol'],
                    title=f'ROE vs 大单均净流入（筛选 {len(merged)} 只）',
                    labels={roe_col: f'ROE (%, {LOOKBACK_DAYS}期)', 'avg_net_flow': '大单均净流入'},
                )
                fig.show()
        else:
            print(f'资金流数据可用列: {ff_all.columns.tolist()}')

---
## Module D — C++ 特征验证（雷达图）

> 需要先修复 `trade_py/ui/services/cpp_bridge.py` 路径

In [ ]:
# 参数
WATCHLIST = []  # 留空则从 instruments 取前10只

from trade_py.ui.services import cpp_bridge

if not cpp_bridge.cli_available():
    print(f'⚠ trade_cli 不可用，路径: {cpp_bridge._TRADE_CLI}')
    print('   请先编译项目: cmake --build build/linux-clang')
else:
    print(f'✅ trade_cli 已就绪: {cpp_bridge._TRADE_CLI}')

    if not WATCHLIST:
        import sqlite3
        db_path = DATA_ROOT / '.metadata' / 'trade.db'
        if db_path.exists():
            with sqlite3.connect(str(db_path)) as conn:
                rows = conn.execute('SELECT symbol FROM instruments LIMIT 10').fetchall()
            WATCHLIST = [r[0] for r in rows]

    print(f'待分析股票: {WATCHLIST}')

In [ ]:
# D-2: 批量获取特征并绘制雷达图
if 'WATCHLIST' in dir() and WATCHLIST and cpp_bridge.cli_available():
    feature_rows = []
    for sym in WATCHLIST:
        result = cpp_bridge.get_features(sym)
        if 'error' not in result:
            feature_rows.append({'symbol': sym, **result})
        else:
            print(f'  {sym}: {result["error"]}')

    if feature_rows:
        # 通用雷达图（取数值列）
        df_feat = pd.DataFrame(feature_rows).set_index('symbol')
        numeric_cols = df_feat.select_dtypes(include='number').columns.tolist()[:8]

        fig = go.Figure()
        for sym, row in df_feat.iterrows():
            vals = [float(row.get(c, 0)) for c in numeric_cols]
            fig.add_trace(go.Scatterpolar(
                r=vals + [vals[0]],
                theta=numeric_cols + [numeric_cols[0]],
                name=sym,
                fill='toself',
                opacity=0.5,
            ))
        fig.update_layout(
            polar=dict(radialaxis=dict(visible=True)),
            title='C++ 特征雷达图（自选股对比）',
        )
        fig.show()

        # 排名表
        if numeric_cols:
            df_rank = df_feat[numeric_cols].copy()
            df_rank['综合均值'] = df_rank.mean(axis=1)
            display(df_rank.sort_values('综合均值', ascending=False))
    else:
        print('未能获取任何特征数据')
else:
    print('（Module D 跳过：trade_cli 不可用或 WATCHLIST 为空）')